[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Advanced-Time-Series/blob/main/Quantlets/Ch_15/ATS_ch15_course_tour/ATS_ch15_course_tour.ipynb)

# ATS_ch15_course_tour

The course in one notebook: one short computation per chapter on real data. HAC for squared S&P 500 returns; random walk against AR(1) for EUR/RON with DM-HLN and Clark-West; a variance break in US GDP growth; a recursive monetary VAR; a local level for US inflation; a Markov-switching mean for US GDP growth; a historical-simulation VaR 1% backtest; local Whittle memory of absolute returns; the business-cycle share of the spectrum of industrial production; split conformal and ACI intervals; Granger tests between the S&P 500 and the BET; SADF for the Nasdaq 100.

*Advanced Time Series Analysis and Forecasting (ATS), Chapter 15: Review and project defence. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/ATS/Chapter_15'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# numpy, scipy and statsmodels are preinstalled in Colab
# !pip install statsmodels

In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)
LightBlue = '#7EA6E0'         # thin background series (e.g. the other countries of a panel): a blue tint, never grey

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import ats_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Advanced-Time-Series/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
# local copy of the Oxford-Man realized library, if present (not distributed; read_omi downloads it)
REALIZED_DIR = next((os.path.join(d, 'data', 'realized') for d in ('.', '..', '../..', '../../..')
                     if os.path.isdir(os.path.join(d, 'data', 'realized'))), '')
END = '2026-09-18'          # last day of the saved data
# Oxford-Man Institute's realized library, version 0.3 (Heber, Lunde, Shephard and Sheppard 2009): the last public
# file (28 February 2022), preserved by the Internet Archive; six indices and the main measures
OMI_URL = ('https://web.archive.org/web/20220301022212id_/'
           'https://realized.oxford-man.ox.ac.uk/images/oxfordmanrealizedvolatilityindices.zip')
OMI_FILE = 'omi_realized_library_v03.csv'
OMI_SYMBOLS = ['.SPX', '.GDAXI', '.FTSE', '.N225', '.STOXX50E', '.FCHI']
OMI_COLS = ['rv5', 'rv5_ss', 'rv10', 'bv', 'bv_ss', 'medrv', 'rk_parzen', 'rk_twoscale', 'rsv', 'open_to_close',
            'close_price', 'open_price', 'nobs']

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the ATS repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_omi(symbols=None, local=True):
    """Oxford-Man Institute's realized library, version 0.3: daily realised measures (OMI_COLS) of six indices
    (OMI_SYMBOLS), columns date, symbol and the measures, sorted by symbol and date. The library is not
    redistributed with the course: a local copy data/realized/omi_realized_library_v03.csv is used if present,
    otherwise the last public file (Internet Archive copy of 28 February 2022) is downloaded once, the six indices
    are extracted and cached in the temporary folder. Cite: Heber, G., Lunde, A., Shephard, N. and Sheppard, K.
    (2009), Oxford-Man Institute's realized library, version 0.3, Oxford-Man Institute, University of Oxford."""
    import tempfile
    import zipfile
    if 'omi' not in _CACHE:
        cache = os.path.join(tempfile.gettempdir(), 'ats_omi')
        cands = ([os.path.join(REALIZED_DIR, OMI_FILE)] if (local and REALIZED_DIR) else []) + [os.path.join(cache, OMI_FILE)]
        path = next((c for c in cands if os.path.exists(c)), '')
        if not path:
            print("Downloading the Oxford-Man Institute's realized library, version 0.3 (Heber, Lunde, Shephard and "
                  "Sheppard 2009), last public file of 28 February 2022, from the Internet Archive")
            os.makedirs(cache, exist_ok=True)
            zpath = os.path.join(cache, 'oxfordmanrealizedvolatilityindices.zip')
            if not os.path.exists(zpath):
                req = urllib.request.Request(OMI_URL, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=300) as r:
                    raw = r.read()
                with open(zpath, 'wb') as f:
                    f.write(raw)
            with zipfile.ZipFile(zpath) as z:
                d = pd.read_csv(z.open(z.namelist()[0]))
            d = d.rename(columns={d.columns[0]: 'date'})
            d['date'] = pd.to_datetime(d['date'].str[:10])
            d = d[d['Symbol'].isin(OMI_SYMBOLS)][['date', 'Symbol'] + OMI_COLS].rename(columns={'Symbol': 'symbol'})
            path = os.path.join(cache, OMI_FILE)
            d.sort_values(['symbol', 'date']).to_csv(path, index=False, float_format='%.6g')
        _CACHE['omi'] = pd.read_csv(path, parse_dates=['date'])
    d = _CACHE['omi']
    if symbols is not None:
        d = d[d['symbol'].isin([symbols] if isinstance(symbols, str) else list(symbols))]
    return d.copy()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
import math
from scipy.signal import lfilter

SEED = 2026

KS = [1, 2, 5, 10, 20, 50]

PS = [50, 100, 250, 500, 1000]

DELTAS = [0.1, 0.2, 0.3]

def nw_lags(T):
    """Newey-West (1994) rule of thumb: floor(4 (T/100)^(2/9)) lags."""
    return int(np.floor(4 * (T / 100) ** (2 / 9)))


def lrv(x, lags=None):
    """Long-run variance of x (Bartlett kernel, Newey-West): gamma_0 + 2 sum_j (1 - j/(L+1)) gamma_j."""
    x = np.asarray(x, float) - np.mean(x)
    T = len(x)
    L = nw_lags(T) if lags is None else lags
    v = x @ x / T
    for j in range(1, L + 1):
        v += 2 * (1 - j / (L + 1)) * (x[j:] @ x[:-j]) / T
    return v


def dm_test(e1, e2, h=1, loss='se'):
    """Diebold-Mariano test of equal accuracy, d_t = L(e1) - L(e2); HAC variance with h-1 lags (at least the NW
    rule), Harvey-Leybourne-Newbold small-sample correction, Student t(P-1) p-value (two-sided)."""
    e1, e2 = np.asarray(e1, float), np.asarray(e2, float)
    d = e1 ** 2 - e2 ** 2 if loss == 'se' else np.abs(e1) - np.abs(e2)
    P = len(d)
    L = max(h - 1, nw_lags(P))
    dm = d.mean() / np.sqrt(lrv(d, L) / P)
    hln = dm * np.sqrt((P + 1 - 2 * h + h * (h - 1) / P) / P)
    return {'dbar': float(d.mean()), 'dm': float(dm), 'hln': float(hln), 'p': float(2 * stats.t.sf(abs(hln), P - 1)),
            'P': P}


def ar1(T, rho, rng, burn=100):
    """Gaussian AR(1) with unit innovation variance (a T-vector)."""
    e = rng.standard_normal(T + burn)
    e[0] /= np.sqrt(1 - rho ** 2)
    return lfilter([1.0], [1.0, -rho], e)[burn:]


def hac_t(y, x):
    """t-statistic of the slope of y on (1, x) with a Newey-West covariance."""
    X = np.column_stack([np.ones(len(x)), x])
    XtXi = np.linalg.inv(X.T @ X)
    b = XtXi @ X.T @ y
    e = y - X @ b
    u = X * e[:, None]
    T = len(y)
    L = nw_lags(T)
    S = u.T @ u
    for j in range(1, L + 1):
        G = u[j:].T @ u[:-j]
        S += (1 - j / (L + 1)) * (G + G.T)
    V = XtXi @ S @ XtXi
    return b[1] / np.sqrt(V[1, 1])


def snooping_rates(K, T=250, rho_x=0.9, common=0.5, reps=1000, seed=0, alpha=0.05):
    """Specification search: y_t = u_t (no predictability, AR(1) with rho 0.3); K candidate predictors x_{k,t-1},
    persistent (AR(1) with rho_x) and correlated through a common factor (share `common` of the variance).
    For each data set: HAC t-statistic of each predictor; report the share of data sets in which
      naive   at least one |t| > 1.96 (the best predictor is reported as if it were the only one tried);
      bonf    at least one p < alpha/K;
      maxt    max |t| above the 95% quantile of max |t| under the null (simulated from the same design: the
              Reality Check / step-down logic with the dependence across predictors kept)."""
    rng = np.random.default_rng(seed)
    tmax = np.empty(reps)
    pmin = np.empty(reps)
    for r in range(reps):
        y = ar1(T + 1, 0.3, rng)[1:]
        f = ar1(T + 1, rho_x, rng)
        ts = np.empty(K)
        for k in range(K):
            x = np.sqrt(common) * f + np.sqrt(1 - common) * ar1(T + 1, rho_x, rng)
            ts[k] = hac_t(y, x[:-1])
        tmax[r] = np.max(np.abs(ts))
        pmin[r] = 2 * stats.norm.sf(tmax[r])
    half = reps // 2                      # critical value from one half, rejection rate on the other half
    crit = np.quantile(tmax[:half], 1 - alpha)
    return {'K': K, 'naive': float(np.mean(pmin < alpha)), 'bonf': float(np.mean(pmin < alpha / K)),
            'maxt': float(np.mean(tmax[half:] > crit)), 'crit_maxt': float(crit)}


def leakage_r2(T=240, n_test=60, P=100, k=5, reps=500, seed=0):
    """Predictor selection with look-ahead: y and P candidate predictors are independent white noise (no
    predictability). The k predictors with the largest |correlation| with y are selected
      leaky   on the whole sample (training and test periods), or
      honest  on the training sample only;
    OLS on the training sample, forecasts on the last n_test periods, out-of-sample R^2 against the training mean."""
    rng = np.random.default_rng(seed)
    out = {'leaky': [], 'honest': []}
    n_tr = T - n_test
    for _ in range(reps):
        y = rng.standard_normal(T)
        X = rng.standard_normal((T, P))
        for lab, rows in (('leaky', slice(0, T)), ('honest', slice(0, n_tr))):
            yc = y[rows] - y[rows].mean()
            Xc = X[rows] - X[rows].mean(0)
            c = np.abs(Xc.T @ yc) / (np.sqrt((Xc ** 2).sum(0)) * np.sqrt(yc @ yc))
            sel = np.argsort(-c)[:k]
            Z = np.column_stack([np.ones(n_tr), X[:n_tr, sel]])
            b = np.linalg.lstsq(Z, y[:n_tr], rcond=None)[0]
            Zt = np.column_stack([np.ones(n_test), X[n_tr:, sel]])
            f = Zt @ b
            bench = y[:n_tr].mean()
            out[lab].append(1 - np.sum((y[n_tr:] - f) ** 2) / np.sum((y[n_tr:] - bench) ** 2))
    return {k_: np.array(v) for k_, v in out.items()}


def dm_power(P, delta, rho=0.3, reps=2000, seed=0, alpha=0.05):
    """Power of the DM-HLN test (two-sided, level alpha) when the loss differential is d_t = delta + u_t, u_t a
    Gaussian AR(1) with coefficient rho and unit variance: delta is the mean gain in standard deviations of d."""
    rng = np.random.default_rng(seed)
    rej = 0
    for _ in range(reps):
        d = delta + ar1(P, rho, rng)
        L = nw_lags(P)
        s = d.mean() / np.sqrt(lrv(d, L) / P)
        s *= np.sqrt((P - 1) / P)
        rej += 2 * stats.t.sf(abs(s), P - 1) < alpha
    return rej / reps


def dm_power_approx(P, delta, rho=0.3, alpha=0.05):
    """Large-sample power: the statistic is approximately N(delta sqrt(P / Omega), 1), Omega = (1+rho)/(1-rho) the
    long-run variance of d (unit variance AR(1))."""
    om = (1 + rho) / (1 - rho)
    m = delta * np.sqrt(P / om)
    z = stats.norm.ppf(1 - alpha / 2)
    return float(stats.norm.sf(z - m) + stats.norm.cdf(-z - m))


def p_required(delta, rho=0.3, power=0.8, alpha=0.05):
    """Out-of-sample length for a given power (large-sample formula): P = Omega ((z_{1-a/2} + z_power) / delta)^2."""
    om = (1 + rho) / (1 - rho)
    return float(om * ((stats.norm.ppf(1 - alpha / 2) + stats.norm.ppf(power)) / delta) ** 2)


def tour_dependence():
    r = log_returns('sp500', start='2000-01-01')
    x = (r ** 2).values
    T = len(x)
    g0 = x.var()
    om = lrv(x, int(1.3 * math.sqrt(T)))
    se_iid, se_nw = math.sqrt(g0 / T), math.sqrt(om / T)
    return dict(T=T, mean=float(x.mean()), ratio=float(om / g0), se_iid=se_iid, se_nw=se_nw)


def tour_forecast(window=60, first='2016-01'):
    s = load_close('eurron')
    y = np.log(s.resample('ME').last()).diff().dropna() * 100
    if s.index[-1] < y.index[-1] - pd.Timedelta(days=3):
        y = y.iloc[:-1]
    t0 = int(np.argmax(y.index >= pd.Timestamp(first)))
    e_rw, e_ar, f_ar = [], [], []
    for t in range(t0, len(y)):
        tr = y.iloc[t - window:t].values
        b = np.linalg.lstsq(np.column_stack([np.ones(window - 1), tr[:-1]]), tr[1:], rcond=None)[0]
        f = b[0] + b[1] * tr[-1]
        e_rw.append(y.iloc[t])
        e_ar.append(y.iloc[t] - f)
        f_ar.append(f)
    e_rw, e_ar, f_ar = map(np.array, (e_rw, e_ar, f_ar))
    cw = e_rw ** 2 - (e_ar ** 2 - f_ar ** 2)
    P = len(cw)
    return dict(P=P, ratio=float(np.sqrt(np.mean(e_ar ** 2) / np.mean(e_rw ** 2))), dm=dm_test(e_ar, e_rw),
                cw=float(cw.mean() / math.sqrt(lrv(cw, nw_lags(P)) / P)))


def sup_wald_mean(x, trim=0.15):
    """sup-Wald statistic for one break in the mean of x (HAC variance under the null) and its date index."""
    x = np.asarray(x, float)
    T = len(x)
    om = lrv(x)
    best, kb = 0.0, None
    for k in range(int(trim * T), int((1 - trim) * T)):
        d = x[:k].mean() - x[k:].mean()
        w = d ** 2 / (om * (1 / k + 1 / (T - k)))
        if w > best:
            best, kb = w, k
    return best, kb


def tour_breaks():
    g = 400 * np.log(read_fred('GDPC1')).diff().dropna().loc['1953':'2019']
    u = np.abs(g - g.mean())
    W, k = sup_wald_mean(u.values)
    return dict(T=len(u), supW=float(W), date=str(u.index[k].date()),
                sd_before=float(g.iloc[:k].std()), sd_after=float(g.iloc[k:].std()))


def tour_svar(p=12):
    from statsmodels.tsa.api import VAR
    f = read_fred(['INDPRO', 'CPIAUCSL', 'FEDFUNDS']).dropna().loc['1965':'2007']
    d = pd.DataFrame({'ip': 100 * np.log(f['INDPRO']).diff(), 'pi': 100 * np.log(f['CPIAUCSL']).diff(),
                      'ff': f['FEDFUNDS']}).dropna()
    res = VAR(d).fit(p)
    irf = res.irf(48).orth_irfs                       # Cholesky, ordering ip, pi, ff
    ip_cum = np.cumsum(irf[:, 0, 2])                  # level response of IP to a funds-rate shock
    return dict(T=int(res.nobs), shock_sd=float(irf[0, 2, 2]), ip_min=float(ip_cum.min()), ip_argmin=int(ip_cum.argmin()),
                pi_max_12=float(np.cumsum(irf[:, 1, 2])[:12].max()))


def tour_state_space():
    from statsmodels.tsa.statespace.structural import UnobservedComponents
    c = read_fred('CPIAUCSL').loc['1990':]
    pi = (1200 * np.log(c).diff()).dropna()
    m = UnobservedComponents(pi.values, level='llevel').fit(disp=False)
    lvl = m.smoothed_state[0]
    return dict(T=len(pi), s2_irr=float(m.params[0]), s2_level=float(m.params[1]), q=float(m.params[1] / m.params[0]),
                level_last=float(lvl[-1]), last=str(pi.index[-1].date()))


def tour_regimes():
    from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
    g = 400 * np.log(read_fred('GDPC1')).diff().dropna().loc['1953':'2019']
    m = MarkovRegression(g.values, k_regimes=2, trend='c', switching_variance=False).fit(search_reps=20)
    mu = m.params[2:4]
    lo = int(np.argmin(mu))
    P = m.regime_transition
    p_ll = float(P[lo, lo, 0])
    prob = m.smoothed_marginal_probabilities[:, lo]
    return dict(mu_low=float(mu[lo]), mu_high=float(mu[1 - lo]), p_low=p_ll, dur_low=1 / (1 - p_ll),
                share_low=float(np.mean(prob > 0.5)))


def tour_var_backtest(window=500, alpha=0.01, start='2010-01-01'):
    r = log_returns('sp500', start='2005-01-01')
    q = r.rolling(window).quantile(alpha).shift(1)
    d = pd.DataFrame({'r': r, 'q': q}).dropna().loc[start:]
    hits = (d['r'] < d['q']).astype(int)
    T, x = len(hits), int(hits.sum())
    ph = x / T
    LR = -2 * ((T - x) * math.log(1 - alpha) + x * math.log(alpha) - (T - x) * math.log(1 - ph) - x * math.log(ph))
    return dict(T=T, hits=x, rate=ph, LR=LR, p=float(stats.chi2.sf(LR, 1)))


def local_whittle(x, m):
    x = np.asarray(x, float) - np.mean(x)
    n = len(x)
    lam = 2 * np.pi * np.arange(1, m + 1) / n
    I = np.abs(np.fft.fft(x)[1:m + 1]) ** 2 / (2 * np.pi * n)

    def R(d):
        return math.log(np.mean(lam ** (2 * d) * I)) - 2 * d * np.mean(np.log(lam))
    return float(optimize.minimize_scalar(R, bounds=(-0.49, 0.99), method='bounded').x)


def tour_long_memory():
    a = np.abs(log_returns('sp500', start='2000-01-01').values)
    n = len(a)
    return {f'm=n^{e}': local_whittle(a, int(n ** e)) for e in (0.5, 0.6, 0.7, 0.8)}


def tour_spectrum():
    ip = 100 * np.log(read_fred('INDPRO').loc['1960':'2019']).diff().dropna()
    x = ip.values - ip.values.mean()
    n = len(x)
    I = np.abs(np.fft.rfft(x)) ** 2 / n
    fr = np.fft.rfftfreq(n)                           # cycles per month
    k = np.ones(9) / 9
    Is = np.convolve(I, k, mode='same')
    per = np.divide(1.0, fr, out=np.full_like(fr, np.inf), where=fr > 0) / 12   # period in years
    bc = (per >= 1.5) & (per <= 8)
    return dict(n=n, share_bc=float(Is[bc].sum() / Is[1:].sum()), peak_years=float(per[1:][np.argmax(Is[1:])]))


def tour_conformal(alpha=0.1, n_cal=500, gamma=0.005):
    r = log_returns('sp500', start='2010-01-01').values
    s = np.abs(r)
    T = len(r)
    q_static = np.quantile(s[:n_cal], math.ceil((n_cal + 1) * (1 - alpha)) / n_cal)
    miss_static = np.abs(r[n_cal:]) > q_static
    a_t, miss = alpha, []
    for t in range(n_cal, T):
        cal = s[t - n_cal:t]
        lev = min(max(1 - a_t, 0.0), 1.0)
        q = np.quantile(cal, lev) if lev < 1 else np.inf
        err = float(abs(r[t]) > q)
        miss.append(err)
        a_t += gamma * (alpha - err)
    miss = np.array(miss)
    return dict(cov_static=float(1 - miss_static.mean()), cov_aci=float(1 - miss.mean()),
                cov_static_2020=float(1 - miss_static[-1000:].mean()), n=T - n_cal)


def tour_granger(p=2):
    d = pd.concat([log_returns('sp500', start='2015-01-01'), log_returns('bet', start='2015-01-01')], axis=1).dropna()
    out = {}
    for y, x in (('bet', 'sp500'), ('sp500', 'bet')):
        Y = d[y].values[p:]
        X = [np.ones(len(Y))] + [d[y].values[p - j:-j] for j in range(1, p + 1)] + [d[x].values[p - j:-j] for j in range(1, p + 1)]
        X = np.column_stack(X)
        XtXi = np.linalg.inv(X.T @ X)
        b = XtXi @ X.T @ Y
        e = Y - X @ b
        u = X * e[:, None]
        L = nw_lags(len(Y))
        S = u.T @ u
        for j in range(1, L + 1):
            G = u[j:].T @ u[:-j]
            S += (1 - j / (L + 1)) * (G + G.T)
        V = XtXi @ S @ XtXi
        idx = list(range(1 + p, 1 + 2 * p))
        W = float(b[idx] @ np.linalg.solve(V[np.ix_(idx, idx)], b[idx]))
        out[f'{x}->{y}'] = dict(W=W, p=float(stats.chi2.sf(W, p)))
    return out


def adf_stat(y):
    dy = np.diff(y)
    X = np.column_stack([np.ones(len(dy)), y[:-1]])
    b, res, *_ = np.linalg.lstsq(X, dy, rcond=None)
    e = dy - X @ b
    s2 = e @ e / (len(dy) - 2)
    return b[1] / math.sqrt(s2 * np.linalg.inv(X.T @ X)[1, 1])


def sadf(y, w0):
    return max(adf_stat(y[:k]) for k in range(w0, len(y) + 1))


def tour_bubbles(reps=199, seed=2026):
    p = load_close('ndx', start='1990-01-01', end='2004-12-31').resample('W-FRI').last().dropna()
    y = np.log(p.values)
    T = len(y)
    r0 = 0.01 + 1.8 / math.sqrt(T)
    w0 = int(r0 * T)
    sadf_path = [adf_stat(y[:k]) for k in range(w0, T + 1)]
    k = int(np.argmax(sadf_path))
    rng = np.random.default_rng(seed)
    null = [sadf(np.cumsum(rng.standard_normal(T)), w0) for _ in range(reps)]   # random walk without drift
    return dict(T=T, w0=w0, sadf=float(max(sadf_path)), date=str(p.index[w0 - 1 + k].date()),
                cv95_mc=float(np.quantile(null, 0.95)), reps=reps)

## Run

In [ ]:
print(tour_dependence())
print(tour_forecast())
print(tour_breaks())
print(tour_svar())
print(tour_state_space())
print(tour_regimes())
print(tour_var_backtest())
print(tour_long_memory())
print(tour_spectrum())
print(tour_conformal())
print(tour_granger())
print(tour_bubbles())